# Notebook 01 — Data Preparation
**Project:** Credit Card Customer Profiling & Segmentation  
**Dataset:** [Kaggle — Credit Card Dataset for Clustering](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)  
**Goal:** Load, inspect, and understand the raw data. Identify quality issues before modelling.

---
| Step | Action |
|------|--------|
| 1 | Load & inspect dataset |
| 2 | Missing value analysis |
| 3 | Statistical summary |
| 4 | Univariate distributions |
| 5 | Frequency feature analysis |
| 6 | Correlation heatmap |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
df.head()

## 1. Dataset Overview

In [ ]:
print(f'Shape       : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Duplicates  : {df.duplicated().sum()}')
print(f'\nColumn types:')
print(df.dtypes)

In [ ]:
df.describe().round(2)

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
m_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})
m_df = m_df[m_df['Count'] > 0]
print(m_df)

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(m_df.index, m_df['Pct (%)'], color='#F44336', edgecolor='white', alpha=0.85)
for bar, v in zip(bars, m_df['Pct (%)']):
    ax.text(v + 0.05, bar.get_y() + bar.get_height()/2, f'{v}%',
            va='center', fontsize=11, fontweight='bold')
ax.set_title('Missing Values by Feature', fontsize=13, fontweight='bold')
ax.set_xlabel('% Missing')
ax.set_xlim(0, m_df['Pct (%)'].max() * 1.4)
plt.tight_layout()
plt.show()
print('Strategy: Impute with column median (robust to right-skewed distributions)')

## 3. Monetary Feature Distributions

In [ ]:
monetary_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS']
palette = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for ax, col, color in zip(axes, monetary_cols, palette):
    ax.hist(df[col].dropna(), bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].median(), color='black', lw=1.5, ls='--',
               label=f'Median: ${df[col].median():,.0f}')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Value ($)')
    ax.legend(fontsize=8)
plt.suptitle('Monetary Feature Distributions — All Right-Skewed', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Observation: Heavy right skew — StandardScaler essential before clustering')

## 4. Frequency Features (0–1 Scale)

In [ ]:
freq_cols = ['BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
             'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT']

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes = axes.flatten()
for ax, col in zip(axes, freq_cols):
    ax.hist(df[col].dropna(), bins=20, color='#2196F3', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Frequency (0–1)')
plt.suptitle('Frequency Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

no_cash = (df['CASH_ADVANCE_FREQUENCY'] == 0).sum()
full_pay = (df['PRC_FULL_PAYMENT'] == 1).sum()
print(f'Customers with zero cash advance frequency : {no_cash:,} ({no_cash/len(df)*100:.1f}%)')
print(f'Customers who always pay in full           : {full_pay:,} ({full_pay/len(df)*100:.1f}%)')

## 5. Correlation Heatmap

In [ ]:
df_num = df.drop(columns=['CUST_ID'])
corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, square=True, ax=ax,
            cbar_kws={'shrink': 0.75}, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_flat = corr.unstack().drop_duplicates()
corr_flat = corr_flat[abs(corr_flat) < 1.0].sort_values(ascending=False)
print('Top 5 positive correlations:')
print(corr_flat.head().round(3))

## Key Findings

| Finding | Detail |
|---|---|
| Dataset size | 8,950 customers, 18 features, 0 duplicates |
| Missing values | `MINIMUM_PAYMENTS` 3.5%, `CREDIT_LIMIT` 0.01% — impute with median |
| Skewness | All monetary features are heavily right-skewed |
| Cash advance | ~50% of customers never take cash advances |
| Full payment | Only ~22% of customers always pay in full |
| High correlation | PURCHASES & PURCHASES_TRX (0.69), CASH_ADVANCE & CASH_ADVANCE_TRX (0.65) |

> **Next:** `02_Feature_Engineering.ipynb` — impute, create 7 ratio features, scale.